# Data Organizer — Option E (Full Lifecycle: D + Models + Benchmarks + Monitoring)

> **Reorganizes the flat `gen_ai_detector_dataset/` directory into Option E format**
> and compresses every dataset folder into a `.tar.lz4` archive.
>
> All source data is archived into **`raw/`** first (immutable originals).
> The remaining pipeline directories are scaffolded empty, ready for future use.
>
> Output path: `data/projects/GAID/`
>
> Set paths in **Cell 2 — Configuration** before running anything else.

| Cell | Operation |
|------|-----------|
| **1** | Imports |
| **2** | ⚙ **Configuration — set paths here** |
| **3** | Archive plan (all source data → `raw/`) |
| **4** | Helper functions (run once) |
| **5** | ▶ Create full directory structure |
| **6** | ▶ Dry run — preview the plan |
| **7** | ▶ Run compression (tar + lz4) |
| **8** | ▶ Generate `README.md` & `CHANGELOG.md` |
| **9** | ▶ Verify — inspect output tree |

### Option E directory layout
```
data/projects/GAID/
├── 01.raw/                      (ALL source archives go here — immutable)
│   ├── fake/
│   ├── real/
│   └── testset/
├── 02.processing/               (intermediate — resized / augmented, SHARED)
│   ├── fake/
│   └── real/
├── 03.processed/                (final — ready to train, VERSIONED)
│   └── v10/
│       ├── fake/
│       └── real/
├── 04.models/                   (trained checkpoints & configs, VERSIONED)
│   ├── v10/
│   └── experimental/
├── 05.benchmarks/               (curated test sets + evaluation results)
│   ├── testset_v1/
│   ├── adversarial/
│   └── results/
├── 06.monitoring/               (production logs, drift, flagged samples)
│   ├── predictions/
│   ├── flagged/
│   ├── drift/
│   └── retraining_candidates/
├── 07.csv/
└── 08.docs/
    ├── README.md
    └── CHANGELOG.md
```

## 1 — Imports

In [1]:
%pip install tqdm --quiet

import json
import os
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path

from tqdm.auto import tqdm

print("✓ imports ready")

Note: you may need to restart the kernel to use updated packages.
✓ imports ready


/home/taiaburrahman/dataset_manager_pro/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2 — Configuration

In [3]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PROJECT — set these before running any other cell                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

VERSION     = "v9"      # ← dataset version (used in processed/, models/, docs)

SOURCE_ROOT = Path("/home/taiaburrahman/dataset_manager_pro/data/processed/")
DEST_ROOT   = Path("/home/taiaburrahman/dataset_manager_pro/data/projects/GAID")

# ── Compression settings ──────────────────────────────────────────────────
LZ4_LEVEL      = 1       # 1 = fastest, 12 = highest compression
SKIP_EXISTING  = True    # skip archives that already exist on disk

# ── Verify source exists, create dest if needed ──────────────────────────
assert SOURCE_ROOT.is_dir(), f"Source not found: {SOURCE_ROOT}"
DEST_ROOT.mkdir(parents=True, exist_ok=True)

print(f"┌─ Option E Organizer (Full Lifecycle) ────────────────────────────┐")
print(f"│  Version: {VERSION:<55}│")
print(f"│  Source : {str(SOURCE_ROOT):<55}│")
print(f"│  Dest   : {str(DEST_ROOT):<55}│")
print(f"│  LZ4    : level {LZ4_LEVEL:<48}│")
print(f"│  Skip   : {str(SKIP_EXISTING):<55}│")
print(f"└──────────────────────────────────────────────────────────────────┘")

┌─ Option E Organizer (Full Lifecycle) ────────────────────────────┐
│  Version: v9                                                     │
│  Source : /home/taiaburrahman/dataset_manager_pro/data/processed │
│  Dest   : /home/taiaburrahman/dataset_manager_pro/data/projects/GAID│
│  LZ4    : level 1                                               │
│  Skip   : True                                                   │
└──────────────────────────────────────────────────────────────────┘


## 3 — Archive Plan

Each tuple maps a source folder to its Option E destination:
`(source_relative_path, dest_relative_path, archive_name)`

In [ ]:
ARCHIVE_PLAN: list[tuple[str, str, str]] = [
    # ─── 01.raw/fake/ — PubDB collections ────────────────────────────────
    ("PubDB_Fake/Shutterstock_Fake",           "01.raw/fake", "Shutterstock_Fake"),
    ("PubDB_Fake/ai_shoes",                    "01.raw/fake", "ai_shoes"),
    ("PubDB_Fake/dalle_rec_Fake",              "01.raw/fake", "dalle_rec_Fake"),
    ("PubDB_Fake/pixelpulse_1800",             "01.raw/fake", "pixelpulse_1800"),
    ("PubDB_Fake/tristanzhang32_Fake",         "01.raw/fake", "tristanzhang32_Fake"),

    # ─── 01.raw/fake/ — other AI-generated collections ───────────────────
    ("ai_image_x_collection_feb25_cleaned",    "01.raw/fake", "ai_image_x_collection_feb25_cleaned"),
    ("genAI",                                  "01.raw/fake", "genAI"),
    ("genAI_2",                                "01.raw/fake", "genAI_2"),
    ("genAI_3",                                "01.raw/fake", "genAI_3"),
    ("genAI_4",                                "01.raw/fake", "genAI_4"),
    ("genAI_5",                                "01.raw/fake", "genAI_5"),
    ("genAI_6",                                "01.raw/fake", "genAI_6"),

    # ─── 01.raw/fake/ — Generated Data 2025 ──────────────────────────────
    ("GenAI_Image_Database/Generated_Data_2025/Gen_samples_DALLE_2_300325",     "01.raw/fake", "Gen_samples_DALLE_2_300325"),
    ("GenAI_Image_Database/Generated_Data_2025/Gen_samples_DALLE_3_300325",     "01.raw/fake", "Gen_samples_DALLE_3_300325"),
    ("GenAI_Image_Database/Generated_Data_2025/Gen_samples_GPT_IMAGE_1_070525", "01.raw/fake", "Gen_samples_GPT_IMAGE_1_070525"),
    ("GenAI_Image_Database/Generated_Data_2025/Gen_samples_Gemini_100325",      "01.raw/fake", "Gen_samples_Gemini_100325"),
    ("GenAI_Image_Database/Generated_Data_2025/Gen_samples_Gemini_150325",      "01.raw/fake", "Gen_samples_Gemini_150325"),
    ("GenAI_Image_Database/Generated_Data_2025/Gen_samples_Gemini_190325",      "01.raw/fake", "Gen_samples_Gemini_190325"),
    ("GenAI_Image_Database/Generated_Data_2025/Gen_samples_Gemini_270325",      "01.raw/fake", "Gen_samples_Gemini_270325"),
    ("GenAI_Image_Database/Generated_Data_2025/Gen_samples_Gemini_280325",      "01.raw/fake", "Gen_samples_Gemini_280325"),
    ("GenAI_Image_Database/Generated_Data_2025/Gen_samples_Grok2_200325",       "01.raw/fake", "Gen_samples_Grok2_200325"),
    ("GenAI_Image_Database/Generated_Data_2025/Gen_samples_Grok3_160325",       "01.raw/fake", "Gen_samples_Grok3_160325"),
    ("GenAI_Image_Database/Generated_Data_2025/Gen_samples_Grok3_180325",       "01.raw/fake", "Gen_samples_Grok3_180325"),
    ("GenAI_Image_Database/Generated_Data_2025/Gen_samples_Grok3_220325",       "01.raw/fake", "Gen_samples_Grok3_220325"),

    # ─── 01.raw/real/ — PubDB collections ────────────────────────────────
    ("PubDB_Real/LandscapeRec12K",             "01.raw/real", "LandscapeRec12K"),
    ("PubDB_Real/Shutterstock_Real",           "01.raw/real", "Shutterstock_Real"),
    ("PubDB_Real/SupRes_aditya",               "01.raw/real", "SupRes_aditya"),
    ("PubDB_Real/birdclef",                    "01.raw/real", "birdclef"),
    ("PubDB_Real/dalle_rec_Real",              "01.raw/real", "dalle_rec_Real"),
    ("PubDB_Real/environmental_scenes",        "01.raw/real", "environmental_scenes"),
    ("PubDB_Real/image20",                     "01.raw/real", "image20"),
    ("PubDB_Real/mit_adobe_5k_Real",           "01.raw/real", "mit_adobe_5k_Real"),
    ("PubDB_Real/pht30k_Real",                 "01.raw/real", "pht30k_Real"),
    ("PubDB_Real/real_shoes",                  "01.raw/real", "real_shoes"),
    ("PubDB_Real/tristanzhang32_Real",         "01.raw/real", "tristanzhang32_Real"),
    ("PubDB_Real/ulid25k",                     "01.raw/real", "ulid25k"),

    # ─── 01.raw/real/ — other authentic collections ──────────────────────
    ("real_faces_multinational_130325",         "01.raw/real", "real_faces_multinational_130325"),
    ("human",                                  "01.raw/real", "human"),
    ("human_2",                                "01.raw/real", "human_2"),
    ("human_3",                                "01.raw/real", "human_3"),
    ("GenAI_Image_Database/Generated_Data_2025/selected_real_samples",           "01.raw/real", "selected_real_samples"),
    ("GenAI_Image_Database/Generated_Data_2025/selected_Real_Samples_phase_2",   "01.raw/real", "selected_Real_Samples_phase_2"),

    # ─── 01.raw/testset/ ─────────────────────────────────────────────────
    ("testset/testset_1",                      "01.raw/testset", "testset_1"),
]

print(f"✓ {len(ARCHIVE_PLAN)} archives defined")
print()

stages = {}
for _, dest_rel, _ in ARCHIVE_PLAN:
    stages[dest_rel] = stages.get(dest_rel, 0) + 1
for stage, count in stages.items():
    print(f"  {stage + '/':<35} {count:>2} archives")

## 4 — Helper Functions

Run this cell once to define all utilities used by the dry-run and compression cells.

In [11]:
@dataclass
class ArchiveResult:
    archive_name: str
    source_path: str
    dest_path: str
    source_size_mb: float = 0.0
    archive_size_mb: float = 0.0
    compression_ratio: float = 0.0
    elapsed_seconds: float = 0.0
    status: str = "pending"
    error: str = ""


def get_dir_size(path: Path) -> int:
    """Total bytes of all files under *path* (recursive)."""
    total = 0
    for entry in path.rglob("*"):
        if entry.is_file():
            total += entry.stat().st_size
    return total


def fmt(size_bytes: float) -> str:
    """Human-readable file size."""
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size_bytes < 1024:
            return f"{size_bytes:.1f} {unit}"
        size_bytes /= 1024
    return f"{size_bytes:.1f} PB"


def compress_tar_lz4(source_dir: Path, output_file: Path, src_size: int,
                      lz4_level: int = 1, poll_interval: float = 0.3) -> None:
    """Create a .tar.lz4 archive with a live progress bar.

    Robust against:
      * stderr buffer fill (drained on background threads),
      * tar SIGPIPE (-13) when lz4 closes its stdin early — we treat that as
        success when lz4 itself finished cleanly and the archive exists.
    """
    import threading

    parent = source_dir.parent
    name = source_dir.name

    tar_cmd = ["tar", "cf", "-", "-C", str(parent), name]
    lz4_cmd = ["lz4", "-z", "-q", f"-{lz4_level}", "-", str(output_file)]

    tar_proc = subprocess.Popen(tar_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    lz4_proc = subprocess.Popen(lz4_cmd, stdin=tar_proc.stdout,
                                 stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    tar_proc.stdout.close()

    tar_err: list[bytes] = []
    lz4_err: list[bytes] = []
    t_drain_tar = threading.Thread(target=lambda: tar_err.append(tar_proc.stderr.read()), daemon=True)
    t_drain_lz4 = threading.Thread(target=lambda: lz4_err.append(lz4_proc.stderr.read()), daemon=True)
    t_drain_tar.start()
    t_drain_lz4.start()

    pbar = tqdm(
        total=src_size, unit="B", unit_scale=True, unit_divisor=1024,
        desc=f"  {name}", leave=False,
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
    )

    try:
        while lz4_proc.poll() is None:
            if output_file.exists():
                written = output_file.stat().st_size
                pbar.n = min(written, src_size)
                pbar.refresh()
            time.sleep(poll_interval)

        if output_file.exists():
            pbar.n = src_size
            pbar.refresh()
    finally:
        pbar.close()

    tar_proc.wait()
    t_drain_tar.join(timeout=5)
    t_drain_lz4.join(timeout=5)

    tar_msg = (tar_err[0] if tar_err else b"").decode(errors="replace").strip()
    lz4_msg = (lz4_err[0] if lz4_err else b"").decode(errors="replace").strip()

    if lz4_proc.returncode != 0:
        raise RuntimeError(f"lz4 failed (exit {lz4_proc.returncode}): {lz4_msg}")

    if tar_proc.returncode != 0:
        sigpipe_ok = (tar_proc.returncode == -13 and output_file.exists())
        if not sigpipe_ok:
            raise RuntimeError(
                f"tar failed (exit {tar_proc.returncode}): {tar_msg}"
            )

    if not output_file.exists():
        raise RuntimeError(f"archive not produced: {output_file} (tar={tar_msg!r}, lz4={lz4_msg!r})")


def run_organizer(
    archive_plan: list[tuple[str, str, str]],
    source_root: Path,
    dest_root: Path,
    *,
    dry_run: bool = False,
    lz4_level: int = 1,
    skip_existing: bool = True,
) -> list[ArchiveResult]:
    """Walk the archive plan, compress each source dir, return results."""

    results: list[ArchiveResult] = []
    total_source = 0
    total_archive = 0
    skipped = 0
    failed = 0

    hdr = "DRY RUN" if dry_run else "COMPRESSING"
    print(f"{'=' * 72}")
    print(f"  {hdr} — {len(archive_plan)} archives  (lz4 level {lz4_level})")
    print(f"{'=' * 72}\n")

    overall = tqdm(archive_plan, desc="Overall", unit="archive",
                   bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]")

    for idx, (src_rel, dest_rel, archive_name) in enumerate(overall, 1):
        source_path = source_root / src_rel
        dest_dir = dest_root / dest_rel
        output_file = dest_dir / f"{archive_name}.tar.lz4"

        result = ArchiveResult(
            archive_name=archive_name,
            source_path=str(source_path),
            dest_path=str(output_file),
        )

        overall.set_postfix_str(archive_name)

        # --- source missing ---
        if not source_path.exists():
            tqdm.write(f"  SKIP (missing)  {archive_name}")
            result.status = "skipped_missing"
            results.append(result)
            skipped += 1
            continue

        # --- already compressed ---
        if skip_existing and output_file.exists():
            existing_size = output_file.stat().st_size
            result.status = "skipped_exists"
            result.archive_size_mb = existing_size / (1024 * 1024)
            results.append(result)
            skipped += 1
            tqdm.write(f"  SKIP (exists)   {archive_name}  ({fmt(existing_size)})")
            continue

        # --- measure source ---
        src_size = get_dir_size(source_path)
        result.source_size_mb = src_size / (1024 * 1024)
        total_source += src_size

        if dry_run:
            result.status = "dry_run"
            results.append(result)
            tqdm.write(f"  {archive_name:<50} {fmt(src_size):>10}  → {dest_rel}/")
            continue

        # --- compress with per-archive progress bar ---
        dest_dir.mkdir(parents=True, exist_ok=True)

        try:
            t0 = time.time()
            compress_tar_lz4(source_path, output_file, src_size, lz4_level=lz4_level)
            elapsed = time.time() - t0

            archive_size = output_file.stat().st_size
            result.archive_size_mb = archive_size / (1024 * 1024)
            result.elapsed_seconds = elapsed
            result.compression_ratio = (archive_size / src_size * 100) if src_size > 0 else 0
            result.status = "success"
            total_archive += archive_size

            tqdm.write(f"  ✓ {archive_name:<42} {fmt(src_size):>10} → {fmt(archive_size):>10}"
                       f"  ({result.compression_ratio:.0f}%)  {elapsed:.0f}s")
        except Exception as e:
            result.status = "failed"
            result.error = str(e)
            failed += 1
            tqdm.write(f"  ✗ {archive_name}  FAILED: {e}")
            if output_file.exists():
                output_file.unlink()

        results.append(result)

    overall.close()

    # ── Summary ───────────────────────────────────────────────────────────
    succeeded = sum(1 for r in results if r.status == "success")
    dry_count = sum(1 for r in results if r.status == "dry_run")

    print(f"\n{'=' * 72}")
    print(f"  SUMMARY")
    print(f"{'=' * 72}")
    if dry_run:
        print(f"  Would compress : {dry_count} archives")
    else:
        print(f"  Succeeded      : {succeeded}")
    print(f"  Skipped        : {skipped}")
    print(f"  Failed         : {failed}")
    print(f"  Total source   : {fmt(total_source)}")
    if not dry_run:
        print(f"  Total archives : {fmt(total_archive)}")
        if total_source > 0:
            print(f"  Overall ratio  : {total_archive / total_source * 100:.1f}%")

    if failed > 0:
        print(f"\n  FAILURES:")
        for r in results:
            if r.status == "failed":
                print(f"    - {r.archive_name}: {r.error}")

    # ── JSON report ───────────────────────────────────────────────────────
    report_path = dest_root / "organize_option_e_report.json"
    report = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "source_root": str(source_root),
        "dest_root": str(dest_root),
        "lz4_level": lz4_level,
        "dry_run": dry_run,
        "summary": {
            "total_jobs": len(archive_plan),
            "succeeded": succeeded,
            "skipped": skipped,
            "failed": failed,
            "total_source_mb": round(total_source / (1024 * 1024), 2),
            "total_archive_mb": round(total_archive / (1024 * 1024), 2),
        },
        "archives": [
            {
                "name": r.archive_name,
                "source": r.source_path,
                "dest": r.dest_path,
                "source_mb": round(r.source_size_mb, 2),
                "archive_mb": round(r.archive_size_mb, 2),
                "ratio_pct": round(r.compression_ratio, 1),
                "elapsed_s": round(r.elapsed_seconds, 1),
                "status": r.status,
            }
            for r in results
        ],
    }
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  Report → {report_path}")

    return results


print("✓ helpers defined")

✓ helpers defined


## 5 — Create Full Directory Structure

Creates **all** Option E directories up front — `raw/`, `processing/`, `processed/`,
`models/`, `benchmarks/`, `monitoring/`, `csv/`, and `docs/`.

In [ ]:
ALL_DIRS = [
    # ── 01.raw (archive destinations — populated by compression step) ─────
    "01.raw/fake",
    "01.raw/real",
    "01.raw/testset",
    # ── 02.processing (empty — future intermediate transforms) ────────────
    "02.processing/fake",
    "02.processing/real",
    # ── 03.processed (empty — future versioned training-ready data) ───────
    f"03.processed/{VERSION}/fake",
    f"03.processed/{VERSION}/real",
    # ── 04.models (empty — future trained checkpoints + configs) ──────────
    f"04.models/{VERSION}",
    "04.models/experimental",
    # ── 05.benchmarks (empty — future curated test sets) ──────────────────
    "05.benchmarks/testset_v1/images/fake",
    "05.benchmarks/testset_v1/images/real",
    "05.benchmarks/adversarial/images/fake",
    "05.benchmarks/adversarial/images/real",
    "05.benchmarks/results",
    # ── 06.monitoring (empty — future production logs & drift) ────────────
    "06.monitoring/predictions",
    "06.monitoring/flagged/false_positives/images",
    "06.monitoring/flagged/false_negatives/images",
    "06.monitoring/drift/feature_distributions",
    "06.monitoring/retraining_candidates/images",
    # ── 07.csv & 08.docs ─────────────────────────────────────────────────
    "07.csv/prompts",
    "08.docs",
]

created = 0
for d in ALL_DIRS:
    p = DEST_ROOT / d
    if not p.exists():
        p.mkdir(parents=True, exist_ok=True)
        created += 1
        print(f"  + {d}/")
    else:
        print(f"  ✓ {d}/  (exists)")

print(f"\n  {created} new directories created, {len(ALL_DIRS) - created} already existed")

## 6 — Dry Run (preview only, no compression)

Run this first to verify the plan. Nothing is written to disk.

In [ ]:
dry_results = run_organizer(
    ARCHIVE_PLAN,
    SOURCE_ROOT,
    DEST_ROOT,
    dry_run=True,
    lz4_level=LZ4_LEVEL,
    skip_existing=SKIP_EXISTING,
)

## 7 — Run Compression

**This will create `.tar.lz4` archives for ~191 GB of data — expect it to take a while.**
Already-compressed archives are skipped by default (`SKIP_EXISTING = True`).

In [ ]:
results = run_organizer(
    ARCHIVE_PLAN,
    SOURCE_ROOT,
    DEST_ROOT,
    dry_run=False,
    lz4_level=LZ4_LEVEL,
    skip_existing=SKIP_EXISTING,
)

## 8 — Generate README.md & CHANGELOG.md

Writes project documentation into `docs/`.

In [ ]:
def _archive_table(plan, label):
    """Markdown table rows for archives going to 01.raw/<label>/."""
    return [
        f"| `{name}.tar.lz4` | `{src}` |"
        for src, dest, name in plan if dest == f"01.raw/{label}"
    ]

def _archive_names(plan, label):
    """Archive names for a label."""
    return [name for _, dest, name in plan if dest == f"01.raw/{label}"]

fake_rows = _archive_table(ARCHIVE_PLAN, "fake")
real_rows = _archive_table(ARCHIVE_PLAN, "real")
test_rows = _archive_table(ARCHIVE_PLAN, "testset")
fake_names = _archive_names(ARCHIVE_PLAN, "fake")
real_names = _archive_names(ARCHIVE_PLAN, "real")
test_names = _archive_names(ARCHIVE_PLAN, "testset")
today = time.strftime("%Y-%m-%d")
NL = chr(10)
V = VERSION

# ── README.md ─────────────────────────────────────────────────────────────
readme = f"""# GAID — GenAI Image Detector Dataset

> Organized using **Option E** (Full Lifecycle) layout.
> Generated on {today}.

## Directory Structure

```
GAID/
├── raw/                    Original archives (immutable, one shared copy)
│   ├── fake/               AI-generated image archives ({len(fake_rows)} archives)
│   ├── real/               Authentic photograph archives ({len(real_rows)} archives)
│   └── testset/            Test set archives ({len(test_rows)} archives)
├── processing/             Intermediate transforms (resized, augmented) — empty
│   ├── fake/
│   └── real/
├── processed/              Final training-ready data (versioned) — empty
│   └── {V}/
├── models/                 Trained model checkpoints & configs — empty
│   ├── {V}/
│   └── experimental/
├── benchmarks/             Curated test sets & evaluation results — empty
│   ├── testset_v1/
│   ├── adversarial/
│   └── results/
├── monitoring/             Production inference logs & drift data — empty
│   ├── predictions/
│   ├── flagged/
│   ├── drift/
│   └── retraining_candidates/
├── csv/                    Dataset manifests & metadata
└── docs/                   This README and CHANGELOG
```

## Archive Format

All archives use `.tar.lz4` compression (tar + LZ4 level {LZ4_LEVEL}).

**Decompress a single archive:**
```bash
lz4 -d archive.tar.lz4 | tar xf -
```

**Decompress all archives in a directory:**
```bash
for f in 01.raw/fake/*.tar.lz4; do lz4 -d "$f" - | tar xf - -C ./output/; done
```

## Data Sources

### 01.raw/fake/ — AI-Generated Images

| Archive | Source Directory |
|---------|-----------------|
{NL.join(fake_rows)}

### 01.raw/real/ — Authentic Photographs

| Archive | Source Directory |
|---------|-----------------|
{NL.join(real_rows)}

### 01.raw/testset/ — Test Sets

| Archive | Source Directory |
|---------|-----------------|
{NL.join(test_rows)}

## Pipeline Stages

| Stage | Purpose | Status |
|-------|---------|--------|
| `01.raw/` | Immutable original archives | **Populated** |
| `02.processing/` | Resized / augmented intermediates | Empty (future) |
| `03.processed/{V}/` | Cleaned & validated, training-ready | Empty (future) |
| `04.models/` | Trained checkpoints (.pt, .onnx) + configs | Empty (future) |
| `05.benchmarks/` | Fixed test sets + evaluation results | Empty (future) |
| `06.monitoring/` | Production predictions, drift, flagged samples | Empty (future) |

## Versions

| Version | Date | Description |
|---------|------|-------------|
| {V} | {today} | Initial Option E organization with all raw data |

---
*Auto-generated by `04.data_organizer.ipynb`*
"""

# ── CHANGELOG.md ──────────────────────────────────────────────────────────
fake_list = NL.join(f"- {n}" for n in fake_names)
real_list = NL.join(f"- {n}" for n in real_names)
test_list = NL.join(f"- {n}" for n in test_names)

changelog = f"""# CHANGELOG — GAID Dataset

All notable changes to this dataset are documented in this file.

## [{V}] — {today}

### Added
- Initial Option E directory structure (raw -> processing -> processed + models + benchmarks + monitoring)
- {len(fake_rows)} fake (AI-generated) archives compressed to `01.raw/fake/` as `.tar.lz4`
- {len(real_rows)} real (authentic) archives compressed to `01.raw/real/` as `.tar.lz4`
- {len(test_rows)} test set archive(s) compressed to `01.raw/testset/` as `.tar.lz4`
- Scaffolded empty directories for 02.processing/, 03.processed/, 04.models/, 05.benchmarks/, 06.monitoring/
- Auto-generated README.md and CHANGELOG.md

### Archives — 01.raw/fake/
{fake_list}

### Archives — 01.raw/real/
{real_list}

### Archives — 01.raw/testset/
{test_list}

### Notes
- All archives are immutable originals stored in `01.raw/`
- Compression: tar + LZ4 (level {LZ4_LEVEL})
- 02.processing/ and 03.processed/ directories are empty, ready for future pipeline stages

---
*Auto-generated by `04.data_organizer.ipynb`*
"""

readme_path = DEST_ROOT / "08.docs" / "README.md"
changelog_path = DEST_ROOT / "08.docs" / "CHANGELOG.md"

readme_path.write_text(readme)
changelog_path.write_text(changelog)

print(f"✓ {readme_path}")
print(f"✓ {changelog_path}")

## 9 — Verify Output Tree

Inspect the full Option E directory structure: archives, scaffolded dirs, and totals.

In [ ]:
OPTION_E_DIRS = ["01.raw", "02.processing", "03.processed", "04.models", "05.benchmarks", "06.monitoring", "07.csv", "08.docs"]
total_size = 0
total_count = 0

for top in OPTION_E_DIRS:
    top_path = DEST_ROOT / top
    if not top_path.exists():
        print(f"\n{top}/  (not created yet)")
        continue

    lz4_files = sorted(top_path.rglob("*.tar.lz4"))
    other_files = sorted(
        f.relative_to(DEST_ROOT) for f in top_path.rglob("*") if f.is_file() and not f.name.endswith(".tar.lz4")
    )
    leaf_dirs = sorted(
        d.relative_to(DEST_ROOT) for d in top_path.rglob("*")
        if d.is_dir() and not list(d.iterdir())
    )

    if lz4_files:
        print(f"\n{top}/  ({len(lz4_files)} archives)")
        for lz4_file in lz4_files:
            rel = lz4_file.relative_to(DEST_ROOT)
            size = lz4_file.stat().st_size
            total_size += size
            total_count += 1
            print(f"  {str(rel):<65} {fmt(size):>10}")
    elif other_files:
        print(f"\n{top}/  ({len(other_files)} files)")
        for f in other_files:
            print(f"  {f}")
    elif leaf_dirs:
        print(f"\n{top}/  (scaffolded, empty)")
        for sd in sorted(set(str(d).split("/")[0] + "/" + str(d).split("/")[1] if len(str(d).split("/")) > 1 else str(d) for d in leaf_dirs)):
            print(f"  {sd}/")
    else:
        print(f"\n{top}/  (empty)")

print(f"\n{'─' * 78}")
print(f"  Total: {total_count} archives, {fmt(total_size)}")

## 10 — Extract Archives (LZ4) — Independent Cell

Self-contained cell to decompress `.tar.lz4` archives from a **source path** to a **destination path**.
No dependency on earlier cells — just set the two paths and run.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  EXTRACT .tar.lz4 — set these two paths, then run                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os
import subprocess
import time
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────
LZ4_SOURCE_PATH = Path("/home/taiaburrahman/dataset_manager_pro/data/projects/GAID/01.raw")
LZ4_DEST_PATH   = Path("/home/taiaburrahman/dataset_manager_pro/data/extracted")

DRY_RUN = True   # True = preview only, False = actually extract

# ── Discover archives ────────────────────────────────────────────────────
assert LZ4_SOURCE_PATH.is_dir(), f"Source not found: {LZ4_SOURCE_PATH}"
LZ4_DEST_PATH.mkdir(parents=True, exist_ok=True)

archives = sorted(LZ4_SOURCE_PATH.rglob("*.tar.lz4"))
print(f"Source : {LZ4_SOURCE_PATH}")
print(f"Dest   : {LZ4_DEST_PATH}")
print(f"Found  : {len(archives)} .tar.lz4 archive(s)\n")

if not archives:
    print("⚠ No .tar.lz4 files found in source path.")
else:
    total_size = 0
    for idx, archive in enumerate(archives, 1):
        rel = archive.relative_to(LZ4_SOURCE_PATH)
        size = archive.stat().st_size
        total_size += size
        size_str = f"{size / (1024**3):.2f} GB" if size >= 1024**3 else f"{size / (1024**2):.1f} MB"

        if DRY_RUN:
            print(f"  [{idx:>3}] {str(rel):<60} {size_str:>10}")
        else:
            dest_subdir = LZ4_DEST_PATH / rel.parent
            dest_subdir.mkdir(parents=True, exist_ok=True)

            print(f"  [{idx:>3}/{len(archives)}] Extracting {rel} ({size_str}) ...", end=" ", flush=True)
            t0 = time.time()

            lz4_proc = subprocess.Popen(
                ["lz4", "-d", str(archive), "-c"],
                stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            )
            tar_proc = subprocess.Popen(
                ["tar", "xf", "-", "-C", str(dest_subdir)],
                stdin=lz4_proc.stdout, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            )
            lz4_proc.stdout.close()
            tar_proc.communicate()

            elapsed = time.time() - t0
            if tar_proc.returncode == 0 and lz4_proc.wait() == 0:
                print(f"✓ ({elapsed:.1f}s)")
            else:
                err = tar_proc.stderr.read().decode().strip() or lz4_proc.stderr.read().decode().strip()
                print(f"✗ FAILED: {err}")

    total_str = f"{total_size / (1024**3):.2f} GB" if total_size >= 1024**3 else f"{total_size / (1024**2):.1f} MB"
    mode = "DRY RUN (set DRY_RUN = False to extract)" if DRY_RUN else "DONE"
    print(f"\n{'─' * 72}")
    print(f"  Total: {len(archives)} archives, {total_str}")
    print(f"  Mode : {mode}")

## 5b — Folder to ZIP

Compress selected folders into `.zip` archives. Set `ZIP_SOURCE` and `ZIP_DEST` below.

In [5]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  FOLDER → ZIP — set source & destination, then run                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import zipfile
import time
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────
ZIP_SOURCE = Path("/home/taiaburrahman/dataset_manager_pro/data/processed/v9_relocated/gen_ai_detector_dataset/")
ZIP_DEST   = Path("/home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v9/gen_ai_detector_dataset/")

ZIP_TOP_LEVEL = True   # True = zip each top-level subfolder separately
                       # False = zip the entire ZIP_SOURCE into one archive

# ── Helpers ──────────────────────────────────────────────────────────────
def get_dir_size_zip(path: Path) -> int:
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())

def fmt_size(size_bytes: float) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size_bytes < 1024:
            return f"{size_bytes:.1f} {unit}"
        size_bytes /= 1024
    return f"{size_bytes:.1f} PB"

def zip_folder(source: Path, output: Path) -> tuple[int, int]:
    """Zip a folder, return (file_count, archive_size)."""
    count = 0
    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for file in sorted(source.rglob("*")):
            if file.is_file():
                zf.write(file, file.relative_to(source.parent))
                count += 1
    return count, output.stat().st_size

# ── Run ──────────────────────────────────────────────────────────────────
assert ZIP_SOURCE.is_dir(), f"Source not found: {ZIP_SOURCE}"
ZIP_DEST.mkdir(parents=True, exist_ok=True)

print(f"Source : {ZIP_SOURCE}")
print(f"Dest   : {ZIP_DEST}")
print(f"Mode   : {'per-subfolder' if ZIP_TOP_LEVEL else 'single archive'}\n")

if ZIP_TOP_LEVEL:
    folders = sorted([d for d in ZIP_SOURCE.iterdir() if d.is_dir()])
    if not folders:
        print("⚠ No subfolders found in source path.")
    else:
        total_src = 0
        total_zip = 0
        for idx, folder in enumerate(folders, 1):
            src_size = get_dir_size_zip(folder)
            total_src += src_size
            out_path = ZIP_DEST / f"{folder.name}.zip"

            print(f"  [{idx:>3}/{len(folders)}] {folder.name:<50} {fmt_size(src_size):>10}", end="  ", flush=True)
            t0 = time.time()
            count, zip_size = zip_folder(folder, out_path)
            elapsed = time.time() - t0
            total_zip += zip_size

            ratio = (zip_size / src_size * 100) if src_size > 0 else 0
            print(f"→ {fmt_size(zip_size):>10}  ({ratio:.0f}%)  {count} files  {elapsed:.1f}s")

        print(f"\n{'─' * 78}")
        print(f"  Total: {len(folders)} folders  |  Source: {fmt_size(total_src)}  →  ZIP: {fmt_size(total_zip)}")
else:
    out_path = ZIP_DEST / f"{ZIP_SOURCE.name}.zip"
    src_size = get_dir_size_zip(ZIP_SOURCE)
    print(f"  Zipping entire folder ({fmt_size(src_size)}) ...", flush=True)
    t0 = time.time()
    count, zip_size = zip_folder(ZIP_SOURCE, out_path)
    elapsed = time.time() - t0
    ratio = (zip_size / src_size * 100) if src_size > 0 else 0
    print(f"  ✓ {out_path.name:<50} {fmt_size(zip_size):>10}  ({ratio:.0f}%)  {count} files  {elapsed:.1f}s")

Source : /home/taiaburrahman/dataset_manager_pro/data/processed/v9_relocated/gen_ai_detector_dataset
Dest   : /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v9/gen_ai_detector_dataset/ 
Mode   : per-subfolder



  [  1/13] PubDB_Fake                                            10.1 GB  →    10.0 GB  (99%)  75577 files  510.6s
  [  2/13] PubDB_Real                                            27.9 GB  

KeyboardInterrupt: 

## 5c — Batch Folders → .tar.lz4 (preserves folder names)

Compresses each folder in `FOLDERS_TO_COMPRESS` from `/home/tajrin/data/` into a `.tar.lz4` archive. Each archive is named after its source folder.

In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BATCH FOLDERS → .tar.lz4 (folder names preserved in archives)        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import subprocess
import time
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────
SOURCE_BASE = Path("/home/tajrin/data")

FOLDERS_TO_COMPRESS = [
    "ai_image_ethinicity_deepfake_website",
    "ai_image_glbgpt_web_testset_march25",
    "ai_image_testset_feb26",
    "ai_image_website_collection_dec25",
    "ai_image_website_jan-feb26",
    "ai_image_x_collection_dec25",
    "ai_image_x_july25",
    "image_benchmark_dec25",
    "image_benchmark_feb26",
]

LZ4_OUTPUT_DIR = Path("/home/taiabur/dataset_manager_pro/data/projects/GAID/06.monitoring/data")
LZ4_LEVEL = 1  # 1 = fastest, 12 = highest compression
SKIP_EXISTING = True

# ── Helpers ──────────────────────────────────────────────────────────────
def fmt_sz(b: float) -> str:
    for u in ("B", "KB", "MB", "GB", "TB"):
        if b < 1024:
            return f"{b:.1f} {u}"
        b /= 1024
    return f"{b:.1f} PB"

# ── Run ──────────────────────────────────────────────────────────────────
LZ4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Source base : {SOURCE_BASE}")
print(f"Destination : {LZ4_OUTPUT_DIR}")
print(f"LZ4 level   : {LZ4_LEVEL}")
print(f"Folders     : {len(FOLDERS_TO_COMPRESS)}")
print(f"{'=' * 72}\n")

total_src = 0
total_arc = 0
succeeded = 0
skipped = 0
failed = 0

for idx, folder_name in enumerate(FOLDERS_TO_COMPRESS, 1):
    folder_path = SOURCE_BASE / folder_name
    output_file = LZ4_OUTPUT_DIR / f"{folder_name}.tar.lz4"

    print(f"[{idx:>2}/{len(FOLDERS_TO_COMPRESS)}] {folder_name}")

    if not folder_path.is_dir():
        print(f"       SKIP (not found: {folder_path})\n")
        skipped += 1
        continue

    if SKIP_EXISTING and output_file.exists():
        arc_size = output_file.stat().st_size
        print(f"       SKIP (exists: {fmt_sz(arc_size)})\n")
        skipped += 1
        continue

    files = list(folder_path.rglob("*"))
    file_count = sum(1 for f in files if f.is_file())
    src_size = sum(f.stat().st_size for f in files if f.is_file())
    total_src += src_size

    print(f"       {file_count} files, {fmt_sz(src_size)}")

    t0 = time.time()
    tar_cmd = ["tar", "cf", "-", "-C", str(SOURCE_BASE), folder_name]
    lz4_cmd = ["lz4", f"-{LZ4_LEVEL}", "-", str(output_file)]

    tar_proc = subprocess.Popen(tar_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    lz4_proc = subprocess.Popen(lz4_cmd, stdin=tar_proc.stdout, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    tar_proc.stdout.close()
    lz4_proc.communicate()
    tar_proc.wait()
    elapsed = time.time() - t0

    if tar_proc.returncode != 0 or lz4_proc.returncode != 0:
        err = tar_proc.stderr.read().decode().strip() or lz4_proc.stderr.read().decode().strip()
        print(f"       ✗ FAILED ({elapsed:.1f}s): {err}\n")
        failed += 1
        if output_file.exists():
            output_file.unlink()
    else:
        arc_size = output_file.stat().st_size
        total_arc += arc_size
        ratio = (arc_size / src_size * 100) if src_size > 0 else 0
        succeeded += 1
        print(f"       ✓ {fmt_sz(src_size)} → {fmt_sz(arc_size)}  ({ratio:.0f}%)  {elapsed:.1f}s\n")

print(f"{'=' * 72}")
print(f"  Succeeded : {succeeded}")
print(f"  Skipped   : {skipped}")
print(f"  Failed    : {failed}")
print(f"  Source     : {fmt_sz(total_src)}")
print(f"  Archives  : {fmt_sz(total_arc)}")

Source base : /home/tajrin/data
Destination : /home/taiabur/dataset_manager_pro/data/projects/GAID/06.monitoring/data
LZ4 level   : 1
Folders     : 9

[ 1/9] ai_image_ethinicity_deepfake_website
       SKIP (exists: 856.3 MB)

[ 2/9] ai_image_glbgpt_web_testset_march25
       SKIP (exists: 301.5 MB)

[ 3/9] ai_image_testset_feb26
       SKIP (exists: 100.3 MB)

[ 4/9] ai_image_website_collection_dec25
       SKIP (exists: 7.6 GB)

[ 5/9] ai_image_website_jan-feb26
       SKIP (exists: 6.7 GB)

[ 6/9] ai_image_x_collection_dec25
       SKIP (exists: 5.7 GB)

[ 7/9] ai_image_x_july25
       SKIP (exists: 785.3 MB)

[ 8/9] image_benchmark_dec25
       SKIP (exists: 323.9 MB)

[ 9/9] image_benchmark_feb26
       306 files, 105.8 MB
       ✓ 105.8 MB → 106.0 MB  (100%)  0.4s

  Succeeded : 1
  Skipped   : 8
  Failed    : 0
  Source     : 105.8 MB
  Archives  : 106.0 MB


## 5d — Auto Compress Subfolders → .tar.lz4

Set a **source path** and **destination path** below. Every immediate subfolder in the source will be compressed into a `.tar.lz4` archive named after the subfolder, written into the destination folder.

In [7]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  AUTO: SUBFOLDERS → .tar.lz4 — set source & destination, then run     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import subprocess
import time
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────
AUTO_SOURCE_PATH     = Path("/home/taiaburrahman/dataset_manager_pro/data/processed/v9_extras")
AUTO_DEST_PATH       = Path("/home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v9_extras/")

AUTO_LZ4_LEVEL      = 1      # 1 = fastest, 12 = highest compression
AUTO_SKIP_EXISTING  = True   # skip archives that already exist
AUTO_DRY_RUN        = False  # True = list subfolders only, no compression

# ── Helpers ──────────────────────────────────────────────────────────────
def _fmt(b: float) -> str:
    for u in ("B", "KB", "MB", "GB", "TB"):
        if b < 1024:
            return f"{b:.1f} {u}"
        b /= 1024
    return f"{b:.1f} PB"

def _dir_size(p: Path) -> tuple[int, int]:
    total = 0
    count = 0
    for f in p.rglob("*"):
        if f.is_file():
            total += f.stat().st_size
            count += 1
    return total, count

# ── Discover subfolders ──────────────────────────────────────────────────
assert AUTO_SOURCE_PATH.is_dir(), f"Source not found: {AUTO_SOURCE_PATH}"
AUTO_DEST_PATH.mkdir(parents=True, exist_ok=True)

subfolders = sorted(d for d in AUTO_SOURCE_PATH.iterdir() if d.is_dir())

print(f"Source : {AUTO_SOURCE_PATH}")
print(f"Dest   : {AUTO_DEST_PATH}")
print(f"LZ4    : level {AUTO_LZ4_LEVEL}")
print(f"Found  : {len(subfolders)} subfolder(s)")
print(f"Mode   : {'DRY RUN' if AUTO_DRY_RUN else 'COMPRESS'}")
print(f"{'=' * 72}\n")

if not subfolders:
    print("⚠ No subfolders found in source path.")
else:
    total_src = 0
    total_arc = 0
    succeeded = 0
    skipped = 0
    failed = 0

    for idx, folder in enumerate(subfolders, 1):
        name = folder.name
        output_file = AUTO_DEST_PATH / f"{name}.tar.lz4"

        print(f"[{idx:>3}/{len(subfolders)}] {name}")

        if AUTO_SKIP_EXISTING and output_file.exists():
            arc_size = output_file.stat().st_size
            print(f"         SKIP (exists: {_fmt(arc_size)})\n")
            skipped += 1
            continue

        src_size, file_count = _dir_size(folder)
        total_src += src_size
        print(f"         {file_count} files, {_fmt(src_size)}")

        if AUTO_DRY_RUN:
            print(f"         → would create {output_file}\n")
            continue

        t0 = time.time()
        tar_cmd = ["tar", "cf", "-", "-C", str(AUTO_SOURCE_PATH), name]
        lz4_cmd = ["lz4", f"-{AUTO_LZ4_LEVEL}", "-", str(output_file)]

        tar_proc = subprocess.Popen(tar_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        lz4_proc = subprocess.Popen(lz4_cmd, stdin=tar_proc.stdout,
                                     stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        tar_proc.stdout.close()
        lz4_proc.communicate()
        tar_proc.wait()
        elapsed = time.time() - t0

        if tar_proc.returncode != 0 or lz4_proc.returncode != 0:
            err = (tar_proc.stderr.read().decode().strip()
                   or lz4_proc.stderr.read().decode().strip())
            print(f"         ✗ FAILED ({elapsed:.1f}s): {err}\n")
            failed += 1
            if output_file.exists():
                output_file.unlink()
        else:
            arc_size = output_file.stat().st_size
            total_arc += arc_size
            ratio = (arc_size / src_size * 100) if src_size > 0 else 0
            succeeded += 1
            print(f"         ✓ {_fmt(src_size)} → {_fmt(arc_size)}  ({ratio:.0f}%)  {elapsed:.1f}s\n")

    print(f"{'=' * 72}")
    print(f"  Succeeded : {succeeded}")
    print(f"  Skipped   : {skipped}")
    print(f"  Failed    : {failed}")
    print(f"  Source    : {_fmt(total_src)}")
    if not AUTO_DRY_RUN:
        print(f"  Archives  : {_fmt(total_arc)}")
        if total_src > 0 and total_arc > 0:
            print(f"  Ratio     : {total_arc / total_src * 100:.1f}%")

Source : /home/taiaburrahman/dataset_manager_pro/data/processed/v9_extras
Dest   : /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v9_extras
LZ4    : level 1
Found  : 2 subfolder(s)
Mode   : COMPRESS

[  1/2] Gen_samples
         10475 files, 96.3 MB
         ✓ 96.3 MB → 95.3 MB  (99%)  6.1s

[  2/2] gen_ai_detector_dataset
         219344 files, 43.0 GB
         ✓ 43.0 GB → 42.7 GB  (99%)  570.2s

  Succeeded : 2
  Skipped   : 0
  Failed    : 0
  Source    : 43.1 GB
  Archives  : 42.8 GB
  Ratio     : 99.2%


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# SINGLE-FOLDER COMPRESS
#   One-off helper: compress a single directory into a .tar.lz4 archive.
#   Reuses the compress_tar_lz4() helper defined in cell 4 (run that first).
# ─────────────────────────────────────────────────────────────────────────
SOURCE_DIR     = Path("/home/taiaburrahman/dataset_manager_pro/data/processed/v9_relocated/Gen_samples")
DEST_DIR       = Path("/home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v9/Gen_samples")
ARCHIVE_NAME   = None       # None → use SOURCE_DIR.name; otherwise override (no extension)
LZ4_LEVEL_ONE  = 1          # 1 = fastest, 12 = highest compression
SKIP_EXISTING  = True       # skip if the .tar.lz4 already exists
DRY_RUN        = False      # True → show plan only, do not write archive

assert SOURCE_DIR.is_dir(), f"Source not found: {SOURCE_DIR}"
DEST_DIR.mkdir(parents=True, exist_ok=True)

archive_stem = ARCHIVE_NAME or SOURCE_DIR.name
output_file  = DEST_DIR / f"{archive_stem}.tar.lz4"

src_bytes = get_dir_size(SOURCE_DIR)
print(f"Source : {SOURCE_DIR}")
print(f"Output : {output_file}")
print(f"Size   : {fmt(src_bytes)}   ({src_bytes:,} bytes)")
print(f"LZ4    : level {LZ4_LEVEL_ONE}     Skip if exists: {SKIP_EXISTING}     Dry-run: {DRY_RUN}")

if DRY_RUN:
    print("\nDry-run only — no archive was written.")
elif output_file.exists() and SKIP_EXISTING:
    existing = output_file.stat().st_size
    print(f"\n⏭  Archive already exists ({fmt(existing)}). Set SKIP_EXISTING=False to overwrite.")
else:
    if output_file.exists():
        output_file.unlink()
    t0 = time.time()
    compress_tar_lz4(SOURCE_DIR, output_file, src_bytes, lz4_level=LZ4_LEVEL_ONE)
    elapsed = time.time() - t0

    archive_bytes = output_file.stat().st_size
    ratio = (1 - archive_bytes / src_bytes) * 100 if src_bytes else 0.0

    print(
        f"\n✓ Done in {elapsed:.1f}s\n"
        f"  Source     : {fmt(src_bytes)}\n"
        f"  Archive    : {fmt(archive_bytes)}\n"
        f"  Saved      : {ratio:.1f} %  (ratio {src_bytes / max(archive_bytes, 1):.2f}x)\n"
        f"  Throughput : {fmt(src_bytes / max(elapsed, 1e-6))}/s\n"
        f"  📦 {output_file}"
    )

Source : /home/taiaburrahman/dataset_manager_pro/data/processed/v9_relocated/gen_ai_detector_dataset/real_faces_multinational_130325
Output : /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v9/gen_ai_detector_dataset/real_faces_multinational_130325.tar.lz4
Size   : 3.7 GB   (3,987,121,282 bytes)
LZ4    : level 1     Skip if exists: True     Dry-run: False



✓ Done in 51.3s
  Source     : 3.7 GB
  Archive    : 3.7 GB
  Saved      : -0.2 %  (ratio 1.00x)
  Throughput : 74.2 MB/s
  📦 /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v9/gen_ai_detector_dataset/real_faces_multinational_130325.tar.lz4
